# Entrenamiento del modelo de detección de phishing (CNN)
**Sistema Predictivo Basado en Deep Learning para la Detección de Phishing en Usuarios Digitales de Arequipa**
Keny Russell Guerra Huanaco — UCSM, Ingeniería de Sistemas — Asesor: Nestor Torres

Este notebook ejecuta el pipeline completo en Google Colab (GPU gratuita, según lo planificado en la Práctica 1/4):
1. Descarga y unifica el dataset (PhiUSIIL, UCI ML Repository — 235,795 URLs reales) balanceado a 80,000 URLs.
2. Preprocesa las URLs (tokenización a nivel de carácter) y genera el split 70/10/20.
3. Entrena la CNN con validación cruzada k-fold (k=10).
4. Exporta el modelo final a ONNX y mide latencia (RNF1: <100ms).

**Antes de empezar:** en el menú superior ve a `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y selecciona **GPU** (T4).


## 1. Instalar dependencias

In [ ]:
!pip install -q ucimlrepo tf2onnx onnx onnxruntime


## 2. Subir el proyecto

Sube el archivo `phishing_detector.zip` (contiene `src/`, `data/`, `models/`, este mismo notebook) que te entregó Claude, y ejecuta la celda siguiente para descomprimirlo.


In [ ]:
from google.colab import files
uploaded = files.upload()  # selecciona phishing_detector.zip


In [ ]:
import zipfile, os
zip_name = [f for f in uploaded.keys() if f.endswith('.zip')][0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
os.chdir('phishing_detector')
print('Directorio actual:', os.getcwd())
!ls -la


## 3. Descargar y unificar el dataset (PhiUSIIL, UCI — ≥80,000 URLs balanceadas)

In [ ]:
!python src/download_dataset.py --n-total 80000 --out data/raw/dataset_unificado.csv


## 4. Preprocesamiento (tokenización a nivel de carácter + split 70/10/20)

In [ ]:
!python src/preprocess.py --in data/raw/dataset_unificado.csv --outdir data/processed --maxlen 200


## 5. Entrenamiento con validación cruzada k-fold (k=10)

**Nota de tiempo:** con GPU T4 de Colab, cada fold entrena en pocos minutos (CNN liviana, ~80,000 URLs). El proceso completo de 10 folds + reentrenamiento final suele tardar entre 20 y 45 minutos. Si tu sesión de Colab tiene límite de cuota, puedes reducir `--k-folds` a 5 como respaldo documentado en el análisis de brechas de la Práctica 4.


In [ ]:
import sys
sys.path.insert(0, 'src')
!python src/train.py --processed-dir data/processed --models-dir models --epochs 15 --batch-size 128 --k-folds 10


## 6. Exportar a ONNX y medir latencia (RNF1: <100ms)

In [ ]:
!python src/export_onnx.py --model models/cnn_phishing_final.keras --out models/cnn_phishing.onnx --processed-dir data/processed


## 7. Revisar métricas finales

In [ ]:
import json
print('--- Promedio k-fold (validación) ---')
print(json.dumps(json.load(open('models/cnn_phishing_kfold_metrics.json'))['promedio'], indent=2, ensure_ascii=False))
print('\n--- Métricas en TEST (nunca visto) ---')
print(json.dumps(json.load(open('models/cnn_phishing_final_test_metrics.json')), indent=2, ensure_ascii=False))
print('\n--- Latencia de inferencia ONNX ---')
print(json.dumps(json.load(open('models/latencia_onnx_metrics.json')), indent=2, ensure_ascii=False))


## 8. Descargar los artefactos finales (modelo ONNX + métricas) a tu computadora

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('modelos_entrenados', 'zip', 'models')
files.download('modelos_entrenados.zip')
